# Quickstart: NOAA HRRR forecast 48 hour - dynamical.org Zarr
A brief introduction to the NOAA HRRR forecast 48 hour dataset transformed into an analysis-ready, cloud-optimized format by dynamical.org.

Dataset documentation: https://dynamical.org/catalog/noaa-hrrr-forecast-48-hour/

In [ ]:
# If running locally, follow README.md for simple dependency installation.
# If using Google Colab, run this cell and then restart the notebook.
%pip install "xarray[complete]>=2025.1.2" "zarr>=3.0.4" requests aiohttp

In [1]:
import xarray as xr

ds = xr.open_zarr("https://data.dynamical.org/noaa/hrrr/forecast-48-hour/latest.zarr?email=optional@email.com", decode_timedelta=True, chunks=None)
ds

<xarray.Dataset> Size: 79TB
Dimensions:                                     (init_time: 10525,
                                                 lead_time: 49, y: 1059, x: 1799)
Coordinates:
    expected_forecast_length                    (init_time) timedelta64[ns] 84kB ...
    ingested_forecast_length                    (init_time) timedelta64[ns] 84kB ...
  * init_time                                   (init_time) datetime64[ns] 84kB ...
    latitude                                    (y, x) float32 8MB ...
  * lead_time                                   (lead_time) timedelta64[ns] 392B ...
    longitude                                   (y, x) float32 8MB ...
    spatial_ref                                 int64 8B ...
    valid_time                                  (init_time, lead_time) datetime64[ns] 4MB ...
  * x                                           (x) float64 14kB -2.698e+06 ....
  * y                                           (y) float64 8kB 1.587e+06 ......
Data variables: (12/20)
    categorical_freezing_rain_surface           (init_time, lead_time, y, x) float32 4TB ...
    categorical_ice_pellets_surface             (init_time, lead_time, y, x) float32 4TB ...
    categorical_rain_surface                    (init_time, lead_time, y, x) float32 4TB ...
    categorical_snow_surface                    (init_time, lead_time, y, x) float32 4TB ...
    composite_reflectivity                      (init_time, lead_time, y, x) float32 4TB ...
    downward_long_wave_radiation_flux_surface   (init_time, lead_time, y, x) float32 4TB ...
    ...                                          ...
    temperature_2m                              (init_time, lead_time, y, x) float32 4TB ...
    total_cloud_cover_atmosphere                (init_time, lead_time, y, x) float32 4TB ...
    wind_u_10m                                  (init_time, lead_time, y, x) float32 4TB ...
    wind_u_80m                                  (init_time, lead_time, y, x) float32 4TB ...
    wind_v_10m                                  (init_time, lead_time, y, x) float32 4TB ...
    wind_v_80m                                  (init_time, lead_time, y, x) float32 4TB ...
Attributes:
    dataset_id:           noaa-hrrr-forecast-48-hour
    dataset_version:      0.1.0
    name:                 NOAA HRRR forecast, 48 hour
    description:          Weather forecasts from the High Resolution Rapid Re...
    attribution:          NOAA NWS NCEP HRRR data processed by dynamical.org ...
    spatial_domain:       CONUS
    spatial_resolution:   3km
    time_domain:          Forecasts initialized 2018-07-13 12:00:00 UTC to Pr...
    time_resolution:      Forecasts initialized every 6 hours.
    forecast_domain:      Forecast lead time 0-48 hours ahead
    forecast_resolution:  Hourly

In [ ]:
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray  # noqa: F401 for .rio accessor
from cmocean import cm 
import seaborn as sns

cubehelix = sns.color_palette("cubehelix", as_cmap=True)
thermal = cm.thermal # others: https://matplotlib.org/cmocean/
# built in matplotlib colormaps: https://matplotlib.org/stable/gallery/color/colormap_reference.html

var = "composite_reflectivity"
init_time = pd.Timestamp("2025-09-04T00")
bounds = (-95, 30, -65, 50)  # lon_min, lat_min, lon_max, lat_max

data = (
    ds[var]
    .sel(init_time=init_time)
    .rio.clip_box(*bounds, crs="EPSG:4326")
    .load()
)

dpi=150
fig, ax = plt.subplots(figsize=(data.x.size/(dpi/2), data.y.size/(dpi/2)), dpi=dpi)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.axis("off")

img = ax.imshow(data.isel(lead_time=0), cmap=cubehelix, vmin=data.min(), vmax=data.percentile(99))
anim = FuncAnimation(fig=fig, frames=data, func=lambda frame: img.set_data(frame), interval=80)

anim.save(f"hrrr_{var}_{init_time:%Y-%m-%dT%H}_{"_".join([str(b) for b in bounds])}.mp4")